Path & Imports 

In [6]:
import os
from pathlib import Path

BASE_DIR = Path("C:/Users/MHEDHBI Farah/Desktop/git/LLM-medical-chatbot")
DATA_DIR = BASE_DIR / "data"

print(f"Base dir: {BASE_DIR}")
print(f"Data dir: {DATA_DIR}")
print(f"Data exists: {DATA_DIR.exists()}")

Base dir: C:\Users\MHEDHBI Farah\Desktop\git\LLM-medical-chatbot
Data dir: C:\Users\MHEDHBI Farah\Desktop\git\LLM-medical-chatbot\data
Data exists: True


Load .env

In [7]:
from dotenv import load_dotenv

env_path = BASE_DIR / ".env"
load_dotenv(dotenv_path=env_path)

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

print(f"Pinecone Key: {'OK' if PINECONE_API_KEY else 'NOT FOUND'}")
print(f"Groq Key:     {'OK' if GROQ_API_KEY else 'NOT FOUND'}")

Pinecone Key: OK
Groq Key:     OK


Load PDF

In [8]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader

def load_pdf_files(data_path):
    loader = DirectoryLoader(
        str(data_path),
        glob="**/*.pdf",
        loader_cls=PyPDFLoader
    )
    return loader.load()

extracted_data = load_pdf_files(DATA_DIR)
print(f"Pages loaded: {len(extracted_data)}")

Pages loaded: 4505


Split Text

In [9]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

def text_split(extracted_data):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20
    )
    return splitter.split_documents(extracted_data)

chunks = text_split(extracted_data)
print(f"Number of chunks: {len(chunks)}")

Number of chunks: 40000


HuggingFace Embeddings

In [10]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector = embeddings.embed_query("Hello world")
print(f"Vector dimension: {len(vector)}")

C:\Users\MHEDHBI Farah\AppData\Local\Temp\ipykernel_21844\1793164203.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Vector dimension: 384


Create Pinecone Index

In [12]:
from dotenv import load_dotenv
import os
from pathlib import Path

BASE_DIR = Path("C:/Users/MHEDHBI Farah/Desktop/git/LLM-medical-chatbot")
env_path = BASE_DIR / ".env"

load_dotenv(dotenv_path=env_path, override=True)

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

print(f"ENV file exists: {env_path.exists()}")
print(f"Pinecone Key found: {'YES - ' + PINECONE_API_KEY[:10] + '...' if PINECONE_API_KEY else 'NO - KEY IS NONE'}")

ENV file exists: True
Pinecone Key found: YES - pcsk_28uNj...


In [13]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=PINECONE_API_KEY)
index_name = "medical-chatbot"

existing_indexes = [i.name for i in pc.list_indexes()]
if index_name not in existing_indexes:
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    print(f"Index '{index_name}' created!")
else:
    print(f"Index '{index_name}' already exists!")

Index 'medical-chatbot' created!


Store in Pinecone

In [14]:
from langchain_pinecone import PineconeVectorStore

vectorstore = PineconeVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    index_name=index_name
)
print("Documents stored in Pinecone successfully!")

Documents stored in Pinecone successfully!


 Test Retrieval

In [15]:
query = "What is diabetes?"
results = vectorstore.similarity_search(query, k=5)

for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Description
Diabetes mellitus is a chronic disease that causes
serious health complications including renal (kidney)
failure, heart disease, stroke, and blindness.
Approximately 17 million Americans have diabetes.
Unfortunately, as many as one-half are unaware they
have it.
Background
Every cell in the human body needs energy in
order to function. The body’s primary energy source
is glucose, a simple sugar resulting from the digestion
of foods containing carbohydrates (sugars and

--- Result 2 ---
Diabetes mellitus
Definition
Diabetes mellitus is a condition in which the pan-
creas no longer produces enough insulin or cells stop
responding to the insulin that is produced, so that
glucose in the blood cannot be absorbed into the
cells of the body. Symptoms include frequent urina-
tion, lethargy, excessive thirst, and hunger. The treat-
ment includes changes in diet, oral medications, and in
some cases, daily injections of insulin.
Description

--- Result 3 ---
GALE ENC